<a href="https://colab.research.google.com/github/ayusha-gif/NIFTY-IV-Ultra-Local-Causal-Surface-Model/blob/main/nifty_iv_ultra_local_causal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NIFTY IV Ultra Local Causal Surface Model




---


> Add blockquote

> Add blockquote





The original approach was inspired by Hagan-West curve-fitting
principles: local interpolation, smooth but not over-smooth surfaces, and stable influence from nearby points only. The first submission (`haganwest_causal_surface`) applied those ideas with `half_life=4, shrink=0.90, clip=0.08` and validated at MSE 0.000074. This notebook is the next iteration — parameters tightened after re-running masked validation under stricter no-lookahead constraints.

---

**Step 1 · Preprocessing**

The dataset is loaded and sorted chronologically. Every NIFTY column name is parsed with a regex to extract the strike price and option type (CE or PE). A boolean mask of all originally-missing IV cells is saved at this point — it's used at the very end to restore observed values exactly and to build the submission file.

**Step 2 · Cross-sectional smile interpolation (same-timestamp, no lookahead)**

For each timestamp independently, the model fits a piecewise-linear IV smile across available strikes. Missing strikes are filled by linear interpolation; strikes outside the observed range are linearly extrapolated using the nearest two endpoints. This is the "base" prediction — it only uses data from the same row, so it has zero temporal leakage by design.

A leave-one-out variant of this same step is run in parallel: each observed cell is predicted by leaving it out and re-fitting from the remaining strikes. This gives a clean in-sample residual for every observed cell without any self-fitting.

**Step 3 · Causal EWM residual correction**

This is the core modelling insight. As the model processes rows in chronological order, it maintains an exponentially-weighted running average of past residuals (observed minus leave-one-out prediction) for each column. When a cell is missing in the current row, the base smile prediction is shifted by the shrunken, clipped version of that running average.

Crucially, the EWM state is updated only *after* the current row's prediction is made — so future rows never inform past predictions. The three parameters control the memory and aggressiveness of this correction:

- `half_life=3` — residual memory decays faster than the previous `half_life=4`, making the model more responsive to recent smile behaviour.
- `residual_shrink=0.85` — residuals are scaled down before applying (regularisation, avoids overcorrecting).
- `residual_clip=0.06` — individual residuals are capped at ±0.06 IV units, preventing outlier quotes from dominating the correction.

**Step 4 · Finish & fallback**

Any cells still missing after the above (e.g. entire strike columns with no history at all) are filled by forward-filling the full IV matrix, then column medians, then clipped to the range [0.01, 5.0]. Finally, all originally-observed cells are restored to their exact raw values — the model only ever touches cells that were genuinely missing.

**Step 5 · Masked validation**

20% of observed cells per column are randomly hidden (across 3 seeds) and treated as if missing. Five configurations are compared. The final parameters (`ultra_h3_s085_c006`) beat all alternatives including the previous submission and the pure linear-smile baseline. PCHIP / monotone cubic was also tested but oversmoothed local quote structure on this dataset.

**Step 6 · Output**

Three files are written — the competition submission in long format (only missing cells), the full filled matrix, and a metadata JSON capturing all parameters and validation results for reproducibility.

In [3]:
from pathlib import Path
import re
import json
import math
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUTPUT_NAME = "submission_ultra_local_causal.csv"
FULL_OUTPUT_NAME = "filled_ultra_local_causal.csv"
METADATA_NAME = "submission_ultra_local_causal_metadata.json"

DATA_CANDIDATES = [
    Path("dataset.csv"),
    Path("/kaggle/input/dataset.csv"),
    Path("/kaggle/input/nifty-iv/dataset.csv"),
    Path(r"C:\Users\theay\Downloads\dataset.csv"),
]
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError("dataset.csv not found. Put it beside this notebook or update DATA_CANDIDATES.")

print("Using data:", DATA_PATH)


Using data: dataset.csv


In [ ]:
df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"], format="%d-%m-%Y %H:%M")
df = df.sort_values("datetime").reset_index(drop=True)

iv_cols = [c for c in df.columns if c.startswith("NIFTY")]
iv_matrix = df[iv_cols].to_numpy(float)
original_missing = np.isnan(iv_matrix)

TICKER_RE = re.compile(r"NIFTY\d{2}[A-Z]{3}\d{2}(\d+)(CE|PE)")
def parse_ticker(col):
    m = TICKER_RE.fullmatch(col)
    if not m:
        raise ValueError(col)
    return int(m.group(1)), m.group(2)

meta = {col: parse_ticker(col) for col in iv_cols}
idx_by_type = {}
strikes_by_type = {}
for typ in ["CE", "PE"]:
    cols = sorted([c for c in iv_cols if meta[c][1] == typ], key=lambda c: meta[c][0])
    idx_by_type[typ] = np.array([iv_cols.index(c) for c in cols], dtype=int)
    strikes_by_type[typ] = np.array([meta[c][0] for c in cols], dtype=float)

print(df.shape, "missing IV cells", int(original_missing.sum()))
print("CE strikes", strikes_by_type["CE"].astype(int).tolist())
print("PE strikes", strikes_by_type["PE"].astype(int).tolist())


## Model

The model uses available strikes at the same timestamp, which is allowed because the IV surface is cross-sectional at each time. It does not use future timestamps. The residual correction is causal: before predicting row `t`, it only knows residuals observed in rows `< t`.


In [ ]:
def linear_extrapolate(x_known, y_known, x_all):
    x = np.asarray(x_known, float)
    y = np.asarray(y_known, float)
    x_all = np.asarray(x_all, float)
    if len(x) == 0:
        return np.full_like(x_all, np.nan, dtype=float)
    if len(x) == 1:
        return np.full_like(x_all, y[0], dtype=float)
    order = np.argsort(x)
    x, y = x[order], y[order]
    pred = np.interp(x_all, x, y)
    left = x_all < x[0]
    if left.any():
        pred[left] = y[0] + (y[1] - y[0]) / (x[1] - x[0]) * (x_all[left] - x[0])
    right = x_all > x[-1]
    if right.any():
        pred[right] = y[-1] + (y[-1] - y[-2]) / (x[-1] - x[-2]) * (x_all[right] - x[-1])
    return pred


def strike_interpolate(values):
    out = values.copy().astype(float)
    for r in range(values.shape[0]):
        for typ in ["CE", "PE"]:
            ind = idx_by_type[typ]
            xs = strikes_by_type[typ]
            row = values[r, ind]
            known = ~np.isnan(row)
            missing = np.isnan(row)
            if missing.any() and known.sum() >= 1:
                pred = linear_extrapolate(xs[known], row[known], xs)
                out[r, ind[missing]] = pred[missing]
    return out


def leave_one_out_strike(values):
    loo = np.full_like(values, np.nan, dtype=float)
    for r in range(values.shape[0]):
        for typ in ["CE", "PE"]:
            ind = idx_by_type[typ]
            xs = strikes_by_type[typ]
            row = values[r, ind]
            known_pos = np.where(~np.isnan(row))[0]
            for pos in known_pos:
                other = known_pos[known_pos != pos]
                if len(other) >= 1:
                    loo[r, ind[pos]] = linear_extrapolate(xs[other], row[other], np.array([xs[pos]]))[0]
    return loo


def finish(pred, train):
    out = pred.copy().astype(float)
    fallback = strike_interpolate(pd.DataFrame(train).ffill().to_numpy(float))
    missing = np.isnan(out)
    out[missing] = fallback[missing]
    med = np.nanmedian(train, axis=0)
    rr, cc = np.where(np.isnan(out))
    out[rr, cc] = med[cc]
    out = np.clip(out, 0.01, 5.0)
    out[~np.isnan(train)] = train[~np.isnan(train)]
    return out


def causal_residual_surface(values, half_life=4, residual_shrink=0.90, residual_clip=0.08):
    base = strike_interpolate(values)
    loo = leave_one_out_strike(values)
    out = base.copy().astype(float)
    residual_ewm = np.full(values.shape[1], np.nan)
    alpha = 1 - 0.5 ** (1 / half_life)

    for r in range(values.shape[0]):
        missing = np.isnan(values[r])
        out[r, missing] = base[r, missing] + np.where(
            np.isnan(residual_ewm[missing]), 0.0, residual_shrink * residual_ewm[missing]
        )

        observed = (~np.isnan(values[r])) & (~np.isnan(loo[r]))
        for c in np.where(observed)[0]:
            residual = float(np.clip(values[r, c] - loo[r, c], -residual_clip, residual_clip))
            residual_ewm[c] = residual if np.isnan(residual_ewm[c]) else alpha * residual + (1 - alpha) * residual_ewm[c]

    return finish(out, values)


## Validation

Masked-cell validation compares the base linear smile with causal residual configurations. PCHIP/monotone cubic tests were considered, but on this dataset they oversmoothed local quote structure and validated worse than the local linear smile.


In [ ]:
def make_mask(values, seed, frac=0.20):
    rng = np.random.default_rng(seed)
    known = ~np.isnan(values)
    val_mask = np.zeros_like(known, dtype=bool)
    for c in range(values.shape[1]):
        rows = np.where(known[:, c])[0]
        n = max(1, int(len(rows) * frac))
        val_mask[rng.choice(rows, n, replace=False), c] = True
    return val_mask


def validate(values, seeds=(11, 22, 33)):
    rows = []
    configs = [
        ("linear_smile", lambda x: finish(strike_interpolate(x), x)),
        ("ultra_h3_s085_c006", lambda x: causal_residual_surface(x, 3, 0.85, 0.06)),
        ("submitted_h4_s090_c008", lambda x: causal_residual_surface(x, 4, 0.90, 0.08)),
        ("causal_h3_s090_c006", lambda x: causal_residual_surface(x, 3, 0.90, 0.06)),
        ("causal_h4_s085_c010", lambda x: causal_residual_surface(x, 4, 0.85, 0.10)),
    ]
    for seed in seeds:
        val_mask = make_mask(values, seed)
        train = values.copy()
        train[val_mask] = np.nan
        for name, fn in configs:
            pred = fn(train)
            mse = float(np.mean((pred[val_mask] - values[val_mask]) ** 2))
            rows.append({"seed": seed, "model": name, "mse": mse, "rmse": math.sqrt(mse)})
    detail = pd.DataFrame(rows)
    summary = detail.groupby("model", as_index=False).agg(mse=("mse", "mean"), rmse=("rmse", "mean")).sort_values("mse")
    return summary, detail

validation_summary, validation_detail = validate(iv_matrix)
print(validation_summary.to_string(index=False))


## Final CSV


In [ ]:
final_iv = causal_residual_surface(iv_matrix, half_life=3, residual_shrink=0.85, residual_clip=0.06)
final_iv[~original_missing] = iv_matrix[~original_missing]

full = pd.DataFrame(final_iv, columns=iv_cols)
full.insert(0, "id", np.arange(len(df)))
full.insert(1, "datetime", df["datetime"].dt.strftime("%d-%m-%Y %H:%M"))
full.insert(2, "underlying_price", df["underlying_price"].to_numpy())
full.to_csv(FULL_OUTPUT_NAME, index=False)

long_rows = []
for r in range(len(df)):
    dt = df.loc[r, "datetime"].strftime("%d-%m-%Y %H:%M")
    for c, col in enumerate(iv_cols):
        if original_missing[r, c]:
            long_rows.append((f"{dt}||{col}", final_iv[r, c]))
submission = pd.DataFrame(long_rows, columns=["id", "value"])
submission.to_csv(OUTPUT_NAME, index=False)

metadata = {
    "data_path": str(DATA_PATH),
    "method": "Ultra-local no-lookahead linear smile interpolation + causal EWM residual correction",
    "hagan_west_principles_used": ["local interpolation", "smoothness without oversmoothing", "stable nearby influence"],
    "lookahead": "No future timestamps are used. Same-timestamp other strikes are used cross-sectionally.",
    "half_life": 3,
    "residual_shrink": 0.85,
    "residual_clip": 0.06,
    "validation_summary": validation_summary.to_dict(orient="records"),
    "rows": int(len(df)),
    "iv_columns": int(len(iv_cols)),
    "missing_iv_cells_filled": int(original_missing.sum()),
    "observed_preserved": bool(np.allclose(final_iv[~original_missing], iv_matrix[~original_missing])),
    "filled_missing_min": float(final_iv[original_missing].min()),
    "filled_missing_max": float(final_iv[original_missing].max()),
}
Path(METADATA_NAME).write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved", OUTPUT_NAME, submission.shape)
print("Saved", FULL_OUTPUT_NAME, full.shape)
print("Missing", int(submission.isna().sum().sum()))
print("Observed preserved", metadata["observed_preserved"])
print("Filled range", metadata["filled_missing_min"], metadata["filled_missing_max"])
print(submission.head().to_string(index=False))
